Author: Iker Gutierrez Fandiño.

# Building the SNS-1064 dataset

This notebook implements a regex-based pipeline to construct a structured clinical QA dataset from TXT guidebooks. The process includes text cleaning, segmentation using regular expressions, filtering, and evaluation.

## 0. Imports and data

Import the required libraries:

In [ ]:
import csv
import re
import pandas as pd

Define the list of clinical guidebook files to be processed.

In [ ]:
files = [
    "ansiedad.txt",
    "atencion_paliativa.txt",
    "cuidados_paliativos_pediatria.txt",
    "diabetes.txt",
    "manejo_ictus.txt",
    "prevencion_secundaria_ictus.txt"
]

## 1. Text cleaning

Each line is normalized to reduce noise introduced during PDF-to-TXT conversion. This includes trimming whitespace and removing leading bullet-like characters.

In [ ]:
def clean_text(t):
    t = t.strip() #To strip leading and trailing whitespaces.
    t = re.sub(r'^[\•\●\-\*]+\s*', '', t)
    t = re.sub(r'^\$?\\bullet\$?\s*', '', t)
    return t.strip() #To apply final whitespace trimming after normalization.

## 2. Text segmentation

Regular expressions are used to detect structural elements in the text, including:

- topic
- question
- subquestion
- judgement
- evidence
- considerations

These patterns drive the rule-based extraction process.

In [ ]:
pregunta_pattern = re.compile(r'^[a-z]\)')
subpregunta_pattern = re.compile(r'^[a-z]\.\d+\.')
contexto_pattern = re.compile(r'^Contexto', re.IGNORECASE)

juicio_pattern = re.compile(r'Juicio:\s*', re.IGNORECASE)
evidencia_pattern = re.compile(r'Evidencia procedente de la investigación:\s*', re.IGNORECASE)
consideraciones_pattern = re.compile(
    r'^(Consideraciones adicionales|Información adicional):\s*',
    re.IGNORECASE
)

tema_start_pattern = re.compile(
    r'^(Pregunta|Pregunta para responder|Pregunta a responder|Subpregunta)',
    re.IGNORECASE
)

## 3. Dataframe creation

The text is processed line by line and segmented into structured QA instances. Segmentation logic relies on local patterns such as:

- "Juicio:"
- "Evidencia procedente de la investigación:"
- "a)", "a.1."

These markers typically appear at the beginning of lines in the TXT files Hence, per-line processing makes it easier to:

- detect section boundaries  
- switch modes (e.g., judgement, evidence)  
- accumulate text until the next trigger appears


Each instance is stored as a row with fields for topic, question, answers, and metadata.

In [ ]:
rows = []

for filename in files:

    with open(filename, encoding="utf8") as f:
        texts = [clean_text(line) for line in f if line.strip()]

    current_tema = ""
    current_pregunta = ""
    current_subpregunta = ""

    juicio = []
    evidencia = []
    consideraciones = []

    mode = None
    collecting_tema = False
    tema_buffer = []

    for text in texts:

        # -------- topic start --------
        if tema_start_pattern.match(text):
            collecting_tema = True
            tema_buffer = []
            continue

        # -------- topic end --------
        if collecting_tema and contexto_pattern.match(text):
            collecting_tema = False
            current_tema = " ".join(tema_buffer).strip()
            continue

        if collecting_tema:
            tema_buffer.append(text)
            continue

        # -------- pregunta --------
        if pregunta_pattern.match(text):

            if juicio or evidencia or consideraciones:
                rows.append({
                    "guidebook": filename,
                    "topic": current_tema,
                    "question": current_pregunta,
                    "subquestion": current_subpregunta,
                    "judgement": " ".join(juicio),
                    "evidence": " ".join(evidencia),
                    "considerations": " ".join(consideraciones)
                })

            current_pregunta = text
            current_subpregunta = ""

            juicio = []
            evidencia = []
            consideraciones = []
            mode = None

            continue

        # -------- subpregunta --------
        if subpregunta_pattern.match(text):

            if juicio or evidencia or consideraciones:
                rows.append({
                    "guidebook": filename,
                    "topic": current_tema,
                    "question": current_pregunta,
                    "subquestion": current_subpregunta,
                    "judgement": " ".join(juicio),
                    "evidence": " ".join(evidencia),
                    "considerations": " ".join(consideraciones)
                })

            current_subpregunta = text

            juicio = []
            evidencia = []
            consideraciones = []
            mode = None

            continue

        # -------- juicio --------
        if juicio_pattern.search(text):
            mode = "judgement"
            text = juicio_pattern.split(text, 1)[1]
            juicio.append(text)
            continue

        # -------- evidencia --------
        if evidencia_pattern.search(text):
            mode = "evidence"
            text = evidencia_pattern.split(text, 1)[1]
            evidencia.append(text)
            continue

        # -------- consideraciones --------
        if consideraciones_pattern.search(text):
            mode = "considerations"
            text = consideraciones_pattern.split(text, 1)[1]
            consideraciones.append(text)
            continue

        # -------- accumulate --------
        if mode == "judgement":
            juicio.append(text)

        elif mode == "evidence":
            evidencia.append(text)

        elif mode == "considerations":
            consideraciones.append(text)

    # -------- save last row --------
    if juicio or evidencia or consideraciones:
        rows.append({
            "guidebook": filename,
            "topic": current_tema,
            "question": current_pregunta,
            "subquestion": current_subpregunta,
            "judgement": " ".join(juicio),
            "evidence": " ".join(evidencia),
            "considerations": " ".join(consideraciones)
        })

df = pd.DataFrame(rows)

## 4. Sample filtering

Noisy samples are removed using regex. In particular, entries with placeholder evidence are excluded: *"ver apartado(s) anterior(es)"*, which means, "see previous section(s)".

In [ ]:
# Regex for skipping samples containing "ver apartado(s) anterior(es)":
skip_pattern = re.compile(r"ver apartado(s)? anterior(es)?", re.IGNORECASE)

filtered_df = df[~df["evidence"].str.contains(skip_pattern, na=False)]

print("Original:", len(df))
print("Filtered:", len(filtered_df))

try:
    filtered_df.to_csv("/content/drive/MyDrive/Master LAP/Corpus_linguistics/dataset_noncurated.csv", index=False, encoding="utf8")
    print("CSV created successfully")
except Exception as e:
    print("Error while saving CSV:", e)

## 5. Intermediate automatic evaluation

Quality checks are performed on the extracted dataset, including:

- Presence rate of optional features
- Presence rate of mandatory features

In [ ]:
#Checkpoint
import re
import pandas as pd

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

filtered_df = pd.read_csv("/content/drive/MyDrive/Master LAP/Corpus_linguistics/dataset_noncurated.csv")

### 5.1 Presence rate of optional features

Let's measure how often optional features (subquestion, considerations) appear in the dataset:

In [ ]:
subq_ratio = filtered_df["subquestion"].notna().mean()
cons_ratio = filtered_df["considerations"].notna().mean()

print("subquestion presence (%):", subq_ratio*100)
print("considerations presence (%):", cons_ratio*100)

### 5.2 Presence rate of mandatory features

Let's verify if all cells of required features (e.g., question, judgement, evidence) are non-empty across all samples:

In [ ]:
required_cols = ["guidebook", "topic", "question", "judgement", "evidence"]

for col in required_cols:
    valid = filtered_df[col].notna() & (filtered_df[col].astype(str).str.strip() != "")
    percentage = valid.mean() * 100
    print(f"{col} presence (%): {percentage:.2f}")

The output cell above shows that there are empty cells in the `judgement` and `evidence` features.

In [ ]:
# True if cell is valid (non-empty, non-null)
valid_mask = (
    filtered_df[required_cols]
    .notna()
    & (filtered_df[required_cols].astype(str).apply(lambda x: x.str.strip() != ""))
).all(axis=1)

# indices of incomplete samples
incomplete_indices = filtered_df.index[~valid_mask].tolist()

print("Number of incomplete samples:", len(incomplete_indices))
print("Indices:")
print(incomplete_indices)

The output cell above shows that there are 11 empty cells in mandatory features. The indices of those samples are provided for later manual curation (§7).

## 6. Train/test split

The dataset is split into training and test sets (80%/20%). Stratification by guidebook ensures proportional representation across domains.

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive', force_remount=True)

df = pd.read_csv("/content/drive/MyDrive/Master LAP/Corpus_linguistics/dataset.csv")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["guidebook"]
)

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

In [ ]:
train_df = train_df.sort_values(by="guidebook").reset_index(drop=True) # sort by guidebook
train_df.to_csv("/content/drive/MyDrive/Master LAP/Corpus_linguistics/train_df.csv", index=False)

test_df = test_df.sort_values(by="guidebook").reset_index(drop=True) # sort by guidebook
test_df.to_csv("/content/drive/MyDrive/Master LAP/Corpus_linguistics/test_df.csv", index=False)

df = pd.concat([train_df, test_df], ignore_index=True)

# sort by guidebook
df = df.sort_values(by="guidebook").reset_index(drop=True)
df.to_csv("/content/drive/MyDrive/Master LAP/Corpus_linguistics/dataset.csv", index=False)

## 7. Manual curation

The entire test set is manually reviewed to fix segmentation errors and ensure data quality. However, the training set is only partially inspected, as noise is less critical in training data and full manual curation would be highly time-consuming.

## 8. Final automatic evaluation

After curation, the dataset is re-evaluated to check that the manual curation worked and now all mandatory features are non-empty.

In [ ]:
#Checkpoint
import re
import pandas as pd

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

df = pd.read_csv("/content/drive/MyDrive/Master LAP/Corpus_linguistics/dataset.csv")

print(f"Dataset size: {len(df)}")

### 8.1 Presence rate of optional features

Let's recompute the distribution of optional fields after manual corrections.

In [ ]:
subq_ratio = df["subquestion"].notna().mean()
cons_ratio = df["considerations"].notna().mean()

print("subquestion presence (%):", subq_ratio*100)
print("considerations presence (%):", cons_ratio*100)

### 8.2. Presence rate of mandatory features

Final evaluation confirms 100% presence for all mandatory features.

In [ ]:
required_cols = ["guidebook", "topic", "question", "judgement", "evidence"]

for col in required_cols:
    valid = df[col].notna() & (df[col].astype(str).str.strip() != "")
    percentage = valid.mean() * 100
    print(f"{col} presence (%): {percentage:.2f}")

In [ ]:
# True if cell is valid (non-empty, non-null)
valid_mask = (
    df[required_cols]
    .notna()
    & (df[required_cols].astype(str).apply(lambda x: x.str.strip() != ""))
).all(axis=1)

# indices of incomplete samples
incomplete_indices = df.index[~valid_mask].tolist()

print("Number of incomplete samples:", len(incomplete_indices))
print("Indices:")
print(incomplete_indices)

## 9. Quantitative analysis

Now that the dataset is ready, let's measure its quantative characteristics, including number of sentences, tokens, and per-feature sentence lengths.

### 9.1. Number of sentences and word tokens

Total sentence and word token counts are computed across all textual features:

In [ ]:
import re
import pandas as pd

def count_sentences(text):
    if pd.isna(text):
        return 0
    sentences = re.split(r'[.!?]+', str(text))
    return len([s for s in sentences if s.strip()])

def count_tokens(text):
    if pd.isna(text):
        return 0
    # extract word tokens (letters + digits, no punctuation)
    tokens = re.findall(r'\b\w+\b', str(text))
    return len(tokens)

# apply over relevant columns
text_cols = ["question", "subquestion", "judgement", "evidence", "considerations"]

df["sentences"] = df[text_cols].fillna("").apply(
    lambda row: sum(count_sentences(x) for x in row), axis=1
)

df["tokens"] = df[text_cols].fillna("").apply(
    lambda row: sum(count_tokens(x) for x in row), axis=1
)

total_sentences = df["sentences"].sum()
total_tokens = df["tokens"].sum()

print("Sentences:", total_sentences)
print("Tokens:", total_tokens)

### 9.2. Sentence length

Average and standard deviation of token lengths is measured for each feature:

In [ ]:
# --- lengths ---
df["q_len"] = df["question"].str.split().str.len()
df["e_len"] = df["evidence"].str.split().str.len()
df["j_len"] = df["judgement"].str.split().str.len()
df["s_len"] = df["subquestion"].str.split().str.len()
df["c_len"] = df["considerations"].str.split().str.len()

# --- question ---
print("Avg question length (tokens):", df["q_len"].mean())
print("Std question length (tokens):", df["q_len"].std())

# --- evidence ---
print("Avg evidence length (tokens):", df["e_len"].mean())
print("Std evidence length (tokens):", df["e_len"].std())

# --- judgement ---
print("Avg judgement length (tokens):", df["j_len"].mean())
print("Std judgement length (tokens):", df["j_len"].std())

# --- subquestion (non-empty only) ---
subq_mask = df["subquestion"].notna() & (df["subquestion"].str.strip() != "")
print("Avg subquestion length (tokens):", df.loc[subq_mask, "s_len"].mean())
print("Std subquestion length (tokens):", df.loc[subq_mask, "s_len"].std())

# --- considerations (non-empty only) ---
cons_mask = df["considerations"].notna() & (df["considerations"].str.strip() != "")
print("Avg considerations length (tokens):", df.loc[cons_mask, "c_len"].mean())
print("Std considerations length (tokens):", df.loc[cons_mask, "c_len"].std())

### 9.3. Samples per guidebook

Distribution of samples across the different clinical guidebooks:

In [ ]:
print("Samples per guidebook:")
print(df["guidebook"].value_counts())